[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/workshops/workshop-01-icfes-csv.ipynb)

# Taller 01 · Preparación de datos ICFES — CSV

**Machine Learning · Semestre 1**

Este taller conserva el ejercicio práctico de clase y añade explicaciones breves para facilitar su estudio.

## Objetivo

Preparar el conjunto de datos ICFES para su uso posterior en Machine Learning, siguiendo este flujo:

1. cargar los datos desde formato **CSV**;
2. separar variables predictoras (`X`) y variable objetivo (`y`);
3. identificar e imputar valores faltantes;
4. transformar variables categóricas;
5. reorganizar el conjunto codificado;
6. separar datos de entrenamiento y prueba.

> **Privacidad:** los outputs originales se retiraron antes de publicar el notebook porque el conjunto contiene la variable `documento`. El código del ejercicio se conserva sin cambios.


## CONTINUACIÓN DE LA CLASE DEL LUNES 07_09_2026

MACHINE LEARNING (PREPARANDO LOS DATOS)

### Aplicación con datos ICFES

## 1. Librerías

- **NumPy**: operaciones numéricas y arreglos.
- **Pandas**: lectura, limpieza y transformación de datos.
- **Matplotlib**: visualización; queda disponible para análisis posteriores.

En esta etapa todavía no entrenamos un modelo: primero preparamos correctamente los datos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

importando el conjunto de datos

## 2. Carga del conjunto de datos

`pd.read_csv()` lee la fuente y crea un `DataFrame`.

Después se revisan dos elementos básicos:
- `shape`: número de filas y columnas;
- `columns`: nombres de las variables.

**Idea clave:** antes de transformar una base, debemos entender su estructura.


In [ ]:
dataset = pd.read_csv('https://cdtiueb.com.co/doctorado/datos_icfes.csv')

print(dataset)
print("\n==============================================")
print("DIMENSIONES DEL CONJUNTO DE DATOS")
print(dataset.shape)

print("\n==============================================")
print("NOMBRES DE LAS COLUMNAS")
print(list(dataset.columns))

## 3. Separar `X` e `y`

En aprendizaje supervisado:

- `X` contiene las **variables predictoras** o características.
- `y` contiene la **variable objetivo** que eventualmente queremos predecir.

El ejercicio conserva la lógica de clase: toma la última columna como objetivo mediante `dataset.columns[-1]`.


In [ ]:
# Se mantiene la misma lógica del notebook original:
# X = todas las variables excepto la última
# y = la última variable del conjunto de datos

target_col = dataset.columns[-1]

X_df = dataset.iloc[:, :-1].copy()
y = dataset.iloc[:, -1].copy()

X = X_df.values

print("============================================== \n data en X")
print(X)

print("============================================== \n data en y")
print(y.values)

print("\nVariable objetivo tomada como última columna:")
print(target_col)

In [ ]:
X

In [ ]:
y

## 4. Valores faltantes e imputación

Un valor faltante (`NaN`) puede impedir que muchos algoritmos trabajen correctamente.

En este ejercicio se utiliza `SimpleImputer`:

- variables numéricas → **media**;
- variables categóricas → **valor más frecuente**.

La imputación no “adivina” el valor verdadero; aplica una regla consistente para completar la matriz de datos.


In [ ]:
from sklearn.impute import SimpleImputer

# Identificar columnas numéricas y categóricas de X
columnas_numericas = X_df.select_dtypes(include=np.number).columns.tolist()
columnas_categoricas = X_df.select_dtypes(exclude=np.number).columns.tolist()

X_df_limpio = X_df.copy()

# Imputar valores faltantes numéricos con la media
for col in columnas_numericas:
    if X_df_limpio[col].isna().all():
        X_df_limpio[col] = X_df_limpio[col].fillna(0)
    elif X_df_limpio[col].isna().any():
        imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
        X_df_limpio[[col]] = imputer.fit_transform(X_df_limpio[[col]])

# Imputar valores faltantes categóricos con el valor más frecuente
for col in columnas_categoricas:
    if X_df_limpio[col].isna().all():
        X_df_limpio[col] = X_df_limpio[col].fillna('DESCONOCIDO')
    elif X_df_limpio[col].isna().any():
        imputer_cat = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
        X_df_limpio[[col]] = imputer_cat.fit_transform(X_df_limpio[[col]]).ravel()

X = X_df_limpio.values

print("Columnas numéricas:")
print(columnas_numericas)

print("\nColumnas categóricas:")
print(columnas_categoricas)

print("\n==============================================")
print("X DESPUÉS DE IMPUTAR VALORES FALTANTES")
print(X)

In [ ]:
X

## CREAR DATAFRAME CON DATOS LIMPIOS

## 5. DataFrame limpio

Después de imputar `X`, se reconstruye un `DataFrame` y se reincorpora `y`.

La comprobación `isnull().sum()` permite verificar cuántos valores faltantes quedan por columna.


In [ ]:
df_limpio = X_df_limpio.copy()
df_limpio[target_col] = y.values

print("--- DATOS CON VALORES FALTANTES INPUTADOS EN LAS VARIABLES X ---")
print(df_limpio)
print('\n')

print("--- CANTIDAD DE VALORES FALTANTES POR COLUMNA ---")
print(df_limpio.isnull().sum())

## CODIFICAR VARIABLES CATEGÓRICAS CON pd.get_dummies

## 6. Codificación de variables categóricas

Los modelos normalmente necesitan entradas numéricas.

`pd.get_dummies()` transforma categorías en columnas indicadoras 0/1.  
`drop_first=True` elimina una categoría de referencia para evitar redundancia perfecta entre las columnas dummy.

Si la variable objetivo es categórica, `LabelEncoder` la convierte a códigos numéricos.


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Separar predictores y variable objetivo
X_limpio_df = df_limpio.drop(columns=[target_col]).copy()
y_limpio = df_limpio[target_col].copy()

# Detectar variables categóricas predictoras
columnas_categoricas = X_limpio_df.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

# Aplicar get_dummies únicamente a las variables predictoras categóricas
df_codificado = pd.get_dummies(
    X_limpio_df,
    columns=columnas_categoricas,
    drop_first=True
)

# Convertir columnas booleanas a int (1 y 0)
for col in df_codificado.columns:
    if df_codificado[col].dtype == 'bool':
        df_codificado[col] = df_codificado[col].astype(int)

# Codificar la variable objetivo si es categórica
target_encoder = None

if (
    y_limpio.dtype == 'object'
    or str(y_limpio.dtype) == 'category'
    or y_limpio.dtype == 'bool'
):
    target_encoder = LabelEncoder()
    y_codificado = target_encoder.fit_transform(y_limpio.astype(str))

    print("--- CODIFICACIÓN DE LA VARIABLE OBJETIVO ---")
    for clase, codigo in zip(
        target_encoder.classes_,
        target_encoder.transform(target_encoder.classes_)
    ):
        print(f"{clase} -> {codigo}")
else:
    y_codificado = y_limpio.values

# Agregar la variable objetivo al final
df_codificado[target_col] = y_codificado

print("\n--- DATOS CODIFICADOS ---")
print(df_codificado)
print('\n')

## REORDENAR COLUMNAS: DUMMY PRIMERO, VARIABLES NUMÉRICAS DESPUÉS Y OBJETIVO AL FINAL

## 7. Reorganizar las columnas

Reordenar columnas no cambia los datos; mejora la legibilidad y deja una estructura clara.

El notebook agrupa:
1. columnas dummy;
2. variables numéricas;
3. variable objetivo al final.

**Detalle de Python:** para seleccionar columnas con `df[nuevo_orden]`, `nuevo_orden` debe ser una lista de **nombres de columnas**, no una `Series` con los valores de una columna.


In [ ]:
# Identificar columnas dummy creadas a partir de variables categóricas
columnas_dummy = [
    col for col in df_codificado.columns
    if col != target_col
    and any(col.startswith(f'{cat}_') for cat in columnas_categoricas)
]

print("Columnas dummy:")
print(columnas_dummy)

# Identificar columnas numéricas que quedaron sin codificar
columnas_numericas_finales = [
    col for col in df_codificado.columns
    if col not in columnas_dummy + [target_col]
]

print("\nColumnas numéricas:")
print(columnas_numericas_finales)

# Crear nuevo orden:
# primero dummy, luego numéricas y la variable objetivo al final
Nuevo_Orden = columnas_dummy + columnas_numericas_finales + [target_col]

# Reordenar el DataFrame
df_reordenado = df_codificado[Nuevo_Orden].copy()

# Usaremos este orden en los pasos siguientes
df_codificado = df_reordenado.copy()

print("\n--- DATOS CODIFICADOS (REORDENADOS) ---")
print(df_codificado)
print('\n')

## VERIFICAR ESTRUCTURA FINAL

## 8. Verificación final

Antes de continuar se comprueba:
- número de filas;
- número de columnas;
- orden de variables;
- nombre de la variable objetivo.

Esta revisión evita entrenar un modelo sobre una matriz mal construida.


In [ ]:
print("--- ESTRUCTURA FINAL DEL DATAFRAME ---")
print(f"🔹 Total de filas: {df_codificado.shape[0]}")
print(f"🔹 Total de columnas: {df_codificado.shape[1]}")
print(f"🔹 Columnas en orden: {list(df_codificado.columns)}")
print(f"🔹 Variable objetivo: {target_col}")
print("\n")

## GUARDAR RESULTADOS

## 9. Guardar el conjunto preparado

El resultado codificado se exporta a CSV para poder reutilizarlo sin repetir toda la preparación.


In [ ]:
df_codificado.to_csv('datos_icfes_CODIFICADO.csv', index=False)
print("✅ Archivo guardado como 'datos_icfes_CODIFICADO.csv'")

Se trabaja con el nuevo conjunto de datos, en este caso `df_codificado`, el cual se asignará al valor de X.

La variable objetivo `y` corresponde a la última columna del conjunto de datos original, siguiendo la misma lógica del notebook de clase.

## 10. Matriz final de predictores y objetivo

Se vuelve a separar:

- `X`: todas las columnas predictoras;
- `y`: la columna objetivo.

En este punto los datos ya están listos para la fase de modelado.


In [ ]:
X = df_codificado.drop(columns=[target_col]).values
y = df_codificado[target_col].values

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)

Separando los datos en conjunto de entrenamiento y conjunto de pruebas

## 11. Entrenamiento y prueba

`train_test_split` divide el conjunto en dos partes:

- **entrenamiento**: datos usados para ajustar un modelo;
- **prueba**: datos reservados para evaluar qué tan bien generaliza.

En el ejercicio:
- `test_size=0.2` → 20 % para prueba;
- `random_state=0` → división reproducible.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

Ahora se muestra el conjunto de datos de entrenamiento con las variables ya preparadas y codificadas

## 12. Inspección de las particiones

Las siguientes celdas muestran `X_train`, `X_test`, `y_train` y `y_test`.

Al ejecutarlas en Colab verás los valores correspondientes a tu sesión. En la versión pública se dejaron los outputs vacíos por privacidad.


In [ ]:
print("Conjunto de entrenamiento:")
print(X_train)

In [ ]:
print("Conjunto de prueba:")
print(X_test)

In [ ]:
print("Variable objetivo del conjunto de entrenamiento:")
print(y_train)

In [ ]:
print("Variable objetivo del conjunto de prueba:")
print(y_test)

El conjunto de datos está aquí:

https://cdtiueb.com.co/doctorado/datos_icfes.csv

---

## Resumen del taller

El flujo completo puede recordarse como:

**cargar → inspeccionar → separar X/y → imputar → codificar → verificar → dividir train/test**

### Preguntas de autoevaluación

1. ¿Por qué un modelo necesita separar `X` e `y`?
2. ¿Qué diferencia existe entre imputar con la media y con la moda?
3. ¿Para qué sirve `pd.get_dummies()`?
4. ¿Qué significa `drop_first=True`?
5. ¿Por qué se reserva un conjunto de prueba?
6. ¿Qué garantiza `random_state=0`?

### Siguiente paso

Después de preparar los datos, el siguiente bloque del curso estudia cómo ajustar modelos de Machine Learning sobre las matrices de entrenamiento y evaluar su comportamiento sobre datos no vistos.
